# 02 — Signal Processing & Quality Filtering

**Goal:** Turn raw PPG segments into clean, validated cardiac-cycle windows ready for feature extraction.

Steps applied to each segment:
1. **Low-pass filter** — remove high-frequency noise
2. **High-pass filter** — remove slow baseline drift
3. **Autocorrelation** — estimate dominant cardiac period
4. **Peak detection (SDET)** — locate systolic peaks and diastolic troughs
5. **Quality check** — reject windows where cycles are inconsistent

**Inputs:** `data/processed/ppg_segments.pkl`

**Outputs:** `data/processed/training_windows.pkl` — dict of {window_id: DataFrame}

---
### Pipeline position
```
01 Import → [02 Signal Processing] → 03 Interpolation → 04 ML Data Loading → 05 Features → 06 Models
```

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import signal as sp_signal

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
REPO_ROOT     = Path.cwd().parent if Path.cwd().name == 'improved' else Path.cwd().parent.parent
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'

SAMPLE_RATE       = 100          # Hz — PPG device sampling rate
WINDOW_SECONDS    = 12           # Minimum segment length to consider (seconds)
WINDOW_SAMPLES    = WINDOW_RATE  = SAMPLE_RATE * WINDOW_SECONDS  # 1200 samples
LOWPASS_CUTOFF    = 8.0          # Hz — removes muscle artefact and high-freq noise
HIGHPASS_CUTOFF   = 0.5          # Hz — removes slow baseline wander
FILTER_ORDER      = 4
MIN_PERFUSION_IDX = 1.0          # Minimum PI to trust the signal

print(f'Window : {WINDOW_SECONDS} s = {WINDOW_SAMPLES} samples at {SAMPLE_RATE} Hz')

In [ ]:
with open(PROCESSED_DIR / 'ppg_segments.pkl', 'rb') as f:
    ppg_segments = pickle.load(f)

total_segs = sum(len(v) for v in ppg_segments.values())
print(f'Loaded {len(ppg_segments)} patients, {total_segs} segments total')

## 1. Signal Filtering

Two Butterworth filters are applied in sequence:
- **Low-pass at 8 Hz** removes high-frequency noise (motion artefact, electrical interference)
- **High-pass at 0.5 Hz** removes slow baseline drift (respiration, motion)

`filtfilt` is used instead of `lfilter` to achieve zero-phase distortion.

In [ ]:
def butter_lowpass(data: np.ndarray, cutoff: float = LOWPASS_CUTOFF,
                   fs: float = SAMPLE_RATE, order: int = FILTER_ORDER) -> np.ndarray:
    """Apply a zero-phase Butterworth low-pass filter."""
    nyq = fs / 2
    b, a = sp_signal.butter(order, cutoff / nyq, btype='low')
    return sp_signal.filtfilt(b, a, data)


def butter_highpass(data: np.ndarray, cutoff: float = HIGHPASS_CUTOFF,
                    fs: float = SAMPLE_RATE, order: int = FILTER_ORDER) -> np.ndarray:
    """Apply a zero-phase Butterworth high-pass filter to remove baseline drift."""
    nyq = fs / 2
    b, a = sp_signal.butter(order, cutoff / nyq, btype='high')
    return sp_signal.filtfilt(b, a, data)


def bandpass_filter(data: np.ndarray) -> np.ndarray:
    """Low-pass then high-pass: isolate the cardiac frequency band."""
    return butter_highpass(butter_lowpass(data))

## 2. Autocorrelation

The autocorrelation of the filtered signal reveals the dominant cardiac period. Its first positive peak after zero-lag gives an estimate of the heartbeat interval in samples.

In [ ]:
def compute_autocorrelation(segment: np.ndarray) -> np.ndarray:
    """
    Compute the normalised autocorrelation of *segment*.

    Returns the positive-lag half of the autocorrelation (lags 0 … N-1).
    """
    seg = segment - segment.mean()
    corr = np.correlate(seg, seg, mode='full')
    corr = corr[corr.size // 2:]          # keep positive lags
    if corr[0] != 0:
        corr = corr / corr[0]             # normalise to 1 at lag-0
    return corr

## 3. Peak Detection — SDET Algorithm

SDET (Successive Decomposition and Extremum) detects systolic peaks (maxima) and diastolic troughs (minima) in the filtered PPG cycle.

The algorithm:
1. Identifies local maxima and minima in the PPG
2. Pairs each minimum–maximum–minimum triplet as one cardiac cycle
3. Validates that exactly one maximum exists between two consecutive minima

In [ ]:
def detect_peaks_sdet(segment: np.ndarray) -> tuple:
    """
    Detect systolic peaks and diastolic troughs using the SDET algorithm.

    Parameters
    ----------
    segment : array of filtered, detrended PPG values

    Returns
    -------
    maxima_idx : indices of systolic peaks
    minima_idx : indices of diastolic troughs
    """
    # scipy find_peaks with minimum distance set to ~0.3 s (30 samples at 100 Hz)
    min_distance = int(0.3 * SAMPLE_RATE)
    maxima_idx, _ = sp_signal.find_peaks(segment, distance=min_distance)
    minima_idx, _ = sp_signal.find_peaks(-segment, distance=min_distance)
    return maxima_idx, minima_idx


def extract_valid_cycles(segment: np.ndarray) -> list:
    """
    Return a list of (min_start, max_peak, min_end) index triplets for each
    valid cardiac cycle found in *segment*.

    A cycle is valid when exactly one systolic peak lies between two
    consecutive diastolic troughs.
    """
    maxima_idx, minima_idx = detect_peaks_sdet(segment)
    cycles = []
    for i in range(len(minima_idx) - 1):
        min_start = minima_idx[i]
        min_end   = minima_idx[i + 1]
        peaks_between = maxima_idx[(maxima_idx > min_start) & (maxima_idx < min_end)]
        if len(peaks_between) == 1:
            cycles.append((min_start, peaks_between[0], min_end))
    return cycles

## 4. Quality Window Selection

A 12-second window is accepted only when its cardiac cycles are mutually consistent:
- Cycle lengths (min-to-min intervals) must not vary by more than 20%
- Amplitude of peaks must not vary by more than 40%
- Correlation of each cycle waveform with the mean cycle must exceed 0.8

In [ ]:
def is_quality_window(cycles: list, segment: np.ndarray,
                      max_length_cv: float = 0.20,
                      max_amplitude_cv: float = 0.40,
                      min_cycle_corr: float = 0.80) -> bool:
    """
    Return True if the cardiac cycles in *segment* are consistent enough
    to be used for feature extraction.

    Parameters
    ----------
    cycles        : list of (min_start, max_peak, min_end) from extract_valid_cycles
    segment       : raw filtered segment values
    max_length_cv : maximum allowed coefficient-of-variation for cycle lengths
    max_amplitude_cv : maximum allowed CV for peak amplitudes
    min_cycle_corr   : minimum mean pairwise correlation of cycle waveforms
    """
    if len(cycles) < 3:
        return False

    lengths    = np.array([c[2] - c[0] for c in cycles], dtype=float)
    amplitudes = np.array([segment[c[1]] - segment[c[0]] for c in cycles], dtype=float)

    if lengths.mean() == 0 or amplitudes.mean() == 0:
        return False
    if (lengths.std() / lengths.mean()) > max_length_cv:
        return False
    if (amplitudes.std() / amplitudes.mean()) > max_amplitude_cv:
        return False

    # Resample each cycle to the median cycle length and check correlation
    median_len = int(np.median(lengths))
    resampled = np.array([
        np.interp(np.linspace(0, 1, median_len),
                  np.linspace(0, 1, c[2] - c[0]),
                  segment[c[0]: c[2]])
        for c in cycles
    ])
    mean_cycle = resampled.mean(axis=0)
    correlations = [
        np.corrcoef(row, mean_cycle)[0, 1] for row in resampled
    ]
    if np.mean(correlations) < min_cycle_corr:
        return False

    return True

## 5. Build Training Windows

Slide a 12-second window over each PPG segment. Filter, find cycles, and keep only windows that pass the quality check.

In [ ]:
def process_segment_into_windows(df: pd.DataFrame, pleth_col: str = 'PLETH') -> dict:
    """
    Slide a 12-second window over *df* and return quality-passing windows.

    Returns a dict of {window_id: DataFrame} where each DataFrame contains
    the filtered PLETH column and the detected cycle indices stored as metadata.
    """
    raw = df[pleth_col].dropna().values
    windows = {}
    step = WINDOW_SAMPLES // 2           # 50 % overlap
    window_id = 0

    for start in range(0, len(raw) - WINDOW_SAMPLES + 1, step):
        chunk = raw[start: start + WINDOW_SAMPLES]
        if np.isnan(chunk).any():
            continue
        filtered = bandpass_filter(chunk)
        cycles   = extract_valid_cycles(filtered)
        if is_quality_window(cycles, filtered):
            window_df = df.iloc[start: start + WINDOW_SAMPLES].copy()
            window_df['PLETH_filtered'] = filtered
            windows[window_id] = window_df
            window_id += 1

    return windows


training_windows = {}
global_id = 0
for pid, segments in ppg_segments.items():
    for seg_df in segments:
        seg_windows = process_segment_into_windows(seg_df)
        for local_id, win_df in seg_windows.items():
            training_windows[global_id] = win_df
            global_id += 1

print(f'Total quality windows : {len(training_windows)}')

## 6. Visualise Sample Windows

In [ ]:
import random
sample_ids = random.sample(list(training_windows.keys()), min(3, len(training_windows)))

for wid in sample_ids:
    win = training_windows[wid]
    raw_pleth      = win['PLETH'].values
    filtered_pleth = win['PLETH_filtered'].values

    fig, axes = plt.subplots(1, 2, figsize=(14, 3))
    axes[0].plot(raw_pleth, color='grey', alpha=0.7, label='Raw')
    axes[0].set_title(f'Window {wid} — raw PPG')
    axes[0].set_xlabel('Sample')
    axes[0].set_ylabel('PLETH')

    axes[1].plot(filtered_pleth, color='steelblue', label='Filtered')
    # Mark detected peaks
    maxima, minima = detect_peaks_sdet(filtered_pleth)
    axes[1].plot(maxima, filtered_pleth[maxima], 'rv', ms=6, label='Systolic peak')
    axes[1].plot(minima, filtered_pleth[minima], 'g^', ms=6, label='Diastolic trough')
    axes[1].set_title(f'Window {wid} — filtered + peaks')
    axes[1].set_xlabel('Sample')
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## 7. Save Training Windows

In [ ]:
with open(PROCESSED_DIR / 'training_windows.pkl', 'wb') as f:
    pickle.dump(training_windows, f)

print(f'Saved {len(training_windows)} windows → data/processed/training_windows.pkl')